In [ ]:
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

PAGEINDEX_API_KEY = "enter-your-pageindex-api-key-here"  


print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")


PageIndex key loaded: ✅


In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)

OPENROUTER_API_KEY="Enter API key here" 

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)
MODEL_NAME = "google/gemini-2.0-flash-001"

# Sanity check — should print your key prefix, not None
print("Client key:", client.api_key[:15], "...")

print("OpenAI key loaded:   ", "✅" if client.api_key    else "❌ Missing!")
print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

Client key: sk-or-v1-c243a8 ...
OpenAI key loaded:    ✅
✅ PageIndex client ready
✅ OpenAI client ready


## Upload and index pdf

In [7]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "/home/srikanth/Desktop/Sahithi/RAG/Vectorless RAG/sample_document.pdf"  

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")

📤 Uploading: /home/srikanth/Desktop/Sahithi/RAG/Vectorless RAG/sample_document.pdf
✅ Uploaded!
📋 Document ID: pi-cmpcwqd2w013q01pn29h20qpw


In [8]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


## Inspect Treee structure

In [9]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "NOTE ON PERSONAL FINANCIAL PLANNING",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This document provides a comprehensive overview of Personal Financial Planning (PFP) for accounting and tax professionals. It outlines the PFP process, covering key phases from reflection and situational analysis to implementation. It details essential management areas, including cash, debt, risk, and wealth, alongside specific planning strategies for taxes, education, retirement, insurance, and estates, while defining the advisor's role and providing financial statement templates.",
  "text": "# NOTE ON PERSONAL FINANCIAL PLANNING\n\nFran\u00e7ois Brouard, DBA, FCPA, FCA\nSprott School of Business, Carleton University\n\nPersonal Financial Planning (PFP) is a growing field of expertise. Therefore, it may be useful to understand the basics as a Chartered Professional Accountant and Tax advisor. This note is prepared to provid

In [10]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] NOTE ON PERSONAL FINANCIAL PLANNING  (p.1)
  └─ [0001] Definition of PFP  (p.2)
  └─ [0002] Motivations for PFP  (p.2)
  └─ [0003] Constraints  (p.2)
  └─ [0004] Life Cycle and PFP  (p.3)
  └─ [0005] Process and Areas of PFP  (p.4)
  └─ [0006] Process - Reflection Phase  (p.5)
  └─ [0007] Process - Situational Analysis Phase  (p.6)
  └─ [0008] Process - Planning and Action Phase - Day-to-Day Management  (p.7)
    └─ [0009] Cash management and Budgeting  (p.7)
    └─ [0010] Assets and Wealth management  (p.7)
      └─ [0011] Steps in purchasing assets  (p.7)
      └─ [0012] Housing  (p.7)
      └─ [0013] Transportation  (p.7)
      └─ [0014] Investments  (p.8)
    └─ [0015] Debt management  (p.10)
    └─ [0016] Risk management  (p.11)
  └─ [0017] Process - Planning and Action Phase - Planning Strategies  (p.12)
    └─ [0018] Tax planning  (p.12)
    └─ [0019] Education Planning  (p.13)
    └─ [0020] Steps in education planning  (p.13)
    └─ [0021] Ste

In [11]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 26
   Each node = one retrievable section of the document


## LLM Tree search

In [12]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [13]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What were the main risk factors?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree, model=MODEL_NAME)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What were the main risk factors?

🧠 LLM Reasoning:
The query asks about 'risk factors'. I need to find the nodes that are most likely to discuss risks. 'Risk management' seems like the most relevant term in the document tree.

Node 0016 is titled 'Risk management'. Therefore, this is highly relevant.

I will add node 0016 to the node_list.

🎯 Selected Node IDs: ['0016']


In [14]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [16]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [17]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [20]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="Give me a summary of the key points in the document?",
    tree=pageindex_tree
)

🔍 Query: Give me a summary of the key points in the document?

🧠 Reasoning: The query asks for a summary of the key points in the document, which suggests that we need to identify nodes that cover the general overview and major sections of the document. Starting from the top,...
🎯 Retrieved node IDs: ['0000', '0001', '0005']
📄 Sections found: ['NOTE ON PERSONAL FINANCIAL PLANNING', 'Definition of PFP', 'Process and Areas of PFP']

📝 Answer:
The document outlines key aspects of Personal Financial Planning (PFP) and details its comprehensive process:

1. **Definition**: PFP is described as an ongoing, integrative process of managing one's financial affairs, involving the development and execution of strategies to meet life goals, taking into account professional, personal, and family situations (Definition of PFP, Page 2).

2. **Phases of PFP**:
   - **Reflection Phase**: Focuses on evaluating personal values, objectives, goals, and priorities (Process and Areas of PFP, Page 4).
   - **S